# 第 46 课｜可复现量化数据流水线

本 Notebook 对应「365 天量化金融」第 46 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里把它重建为默认离线、可核对、可重复执行的量化教学实现。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/46_youtube-iePGy3ipV_0/index.html`
- 环境：`pkm-quant`
- 数据：离线模拟行情；每阶段输出状态和哈希
- 状态：可运行学习稿；不代表策略收益承诺

> 网络接口和研究计算分离：Notebook 默认不请求第三方服务。真实接口只作为适配层示意，计算使用带来源声明的本地快照，避免限流、字段变化和历史数据重算破坏复现。

## 本节量化问题

把脚本拆成函数不等于可靠流水线。稳定边界是：

```text
config → fetch → clean/validate → analyze → plot → manifest
```

每个阶段职责单一，由数据契约连接；一个标的失败时保留其状态，不能伪装成整批成功。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


## 1. 配置是输入，不应散落在代码中

配置集中记录标的、日期、价格字段和输出版本。哈希把配置内容压缩成可追踪标识；内容变化时哈希也变化。

In [2]:
# 创建稳定排序的配置，并计算 SHA-256。
import hashlib

config = {
    "symbols": ["BANK_A", "EV_A", "MISSING_A"],
    "start": "2024-01-01",
    "end": "2025-12-31",
    "price_field": "adj_close",
    "pipeline_version": 1,
}
config_text = json.dumps(config, ensure_ascii=False, sort_keys=True)
config_hash = hashlib.sha256(config_text.encode()).hexdigest()
display(pd.Series({**config, "config_hash": config_hash[:12]}, name="value").to_frame())

,value
symbols,"[BANK_A, EV_A, MISSING_A]"
start,2024-01-01
end,2025-12-31
price_field,adj_close
pipeline_version,1
config_hash,916c4c4a9728


## 2. 四个函数只通过明确 DataFrame 契约连接

- `fetch` 只读取快照并按标的筛选；
- `clean` 检查必需列、唯一键、排序和正价格；
- `analyze` 计算区间收益、最大回撤和观测数；
- `plot` 只消费分析阶段已经生成的净值表。

In [3]:
# 实现职责单一的四阶段函数。
def fetch_snapshot(symbol):
    frame = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
    result = frame.loc[frame["symbol"].eq(symbol)].copy()
    if result.empty:
        raise LookupError(f"{symbol}: empty snapshot")
    return result

def clean_bars(frame):
    required = {"date", "symbol", "adj_close"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"missing columns: {sorted(missing)}")
    result = frame.sort_values(["symbol", "date"]).reset_index(drop=True)
    assert not result.duplicated(["symbol", "date"]).any()
    assert result["adj_close"].gt(0).all()
    return result

def analyze_bars(frame):
    prices = frame.set_index("date")["adj_close"]
    nav = prices / prices.iloc[0]
    drawdown = nav / nav.cummax() - 1
    metrics = {
        "rows": len(frame),
        "start": str(frame["date"].min().date()),
        "end": str(frame["date"].max().date()),
        "interval_return": float(nav.iloc[-1] - 1),
        "max_drawdown": float(drawdown.min()),
    }
    return metrics, pd.DataFrame({"nav": nav, "drawdown": drawdown})

def plot_analysis(result_table, symbol):
    axis = result_table["nav"].plot(figsize=(8, 3), title=f"{symbol} 模拟净值")
    axis.set(xlabel="日期", ylabel="净值")
    axis.grid(alpha=0.2)
    figure = axis.figure
    figure.tight_layout()
    output = OUTPUT_DIR / f"46_{symbol}_nav.png"
    figure.savefig(output, dpi=140)
    plt.close(figure)
    return output

## 3. 编排器记录逐标的证据

`run_one` 负责串联阶段和捕获错误，返回统一状态。输出表同时包含成功和失败资产，便于断点续跑。

In [4]:
# 批量运行；单个空资产只标记失败，不影响其余结果。
def run_one(symbol):
    try:
        raw = fetch_snapshot(symbol)
        clean = clean_bars(raw)
        metrics, result_table = analyze_bars(clean)
        chart = plot_analysis(result_table, symbol)
        return {"symbol": symbol, "status": "ok", **metrics, "chart": chart.name, "error": ""}
    except Exception as exc:
        return {"symbol": symbol, "status": "failed", "error": str(exc)}

statuses = pd.DataFrame([run_one(symbol) for symbol in config["symbols"]])
display(statuses)
assert statuses["status"].tolist() == ["ok", "ok", "failed"]
assert statuses.loc[statuses["status"].eq("ok"), "rows"].eq(756).all()

,symbol,status,rows,start,end,interval_return,max_drawdown,chart,error
0,BANK_A,ok,756.0,2023-01-03,2025-11-25,0.281476,-0.210284,46_BANK_A_nav.png,
1,EV_A,ok,756.0,2023-01-03,2025-11-25,0.468290,-0.300514,46_EV_A_nav.png,
2,MISSING_A,failed,NaN,NaN,NaN,NaN,NaN,NaN,MISSING_A: empty snapshot


## 4. Manifest 让结果可重跑

清单保存配置哈希、阶段状态和输入数据哈希。这里的输入是固定快照；如果快照内容变化，即使文件名相同也会被发现。

In [5]:
# 保存流水线清单；JSON 使用稳定字段，便于后续比较。
input_path = DATA_DIR / "daily_prices.csv"
manifest = {
    "pipeline": "lesson-46-offline",
    "config": config,
    "config_sha256": config_hash,
    "input_sha256": hashlib.sha256(input_path.read_bytes()).hexdigest(),
    "status": statuses.fillna("").to_dict(orient="records"),
}
manifest_path = OUTPUT_DIR / "46_pipeline_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("已写入:", manifest_path)
assert manifest_path.exists()
assert len(manifest["input_sha256"]) == 64

已写入: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain/outputs/46_pipeline_manifest.json


## 方法边界与复用结论


- 单一职责不是函数越小越好；边界应围绕稳定的数据语义。
- 图表和指标必须消费同一结果表，避免展示与计算口径分叉。
- 真实批处理还需重试、日志、缓存、增量水位和失败恢复。


本课代码验证的是方法和数据契约，不是未来收益。迁移到真实研究前，还需重新核对来源授权、复权、时区、可得时间、交易成本和样本外边界。